<a href="https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/14_online_evals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 14 · Online evals and closing the loop

Everything so far has been **offline**: a dataset you curated, run when you decided to run it.
Offline evals answer "did this change break anything I already knew about?"

They cannot answer "is it working right now?", because your dataset only contains cases you have
already thought of. **Online evals** score live traffic as it happens — and the traffic contains the
cases you have not thought of yet.

**New in this lesson:** rules on a tracing project, backfills, sampling and spend limits, annotation
queues, and using human corrections to align a judge

> **Need a key?** You need a LangSmith API key stored in Colab Secrets (🔑 in the left
> sidebar) as `LANGSMITH_API_KEY`, with **"Notebook access" turned on**. If you have not done
> that yet, run **[00 · Setup](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/00_setup.ipynb)** first — it takes 10 minutes and
> checks everything.

In [ ]:
# --- snippet:setup v1 ---
%pip install -qq --progress-bar off \
  "deepagents~=0.7.6" \
  "langchain~=1.3.15" \
  "langchain-openai~=1.5.1" \
  "langsmith~=0.11.0"

import os

try:
    from google.colab import userdata

    key = userdata.get("LANGSMITH_API_KEY")
except Exception:  # not on Colab, or secret unavailable
    from getpass import getpass

    key = os.environ.get("LANGSMITH_API_KEY") or getpass("LANGSMITH_API_KEY: ")

os.environ["LANGSMITH_API_KEY"] = key
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "lcw-14-online-evals"

# One constant, used everywhere. Models are served by the LangSmith gateway,
# so this key is the only credential the notebook needs.
MODEL = "langsmith:openai/gpt-5.6-luna"
# --- /snippet ---

print("Ready.")

In [ ]:
#@title Connect to the shared deployment (run me) { display-mode: "form" }
# --- snippet:remote_agent v1 ---
import httpx
from langgraph.pregel.remote import RemoteGraph

HOST_API = "https://api.host.langchain.com"


def find_deployment(name: str = "support-agent") -> dict:
    """The shared deployment's record, looked up by name."""
    response = httpx.get(
        f"{HOST_API}/v2/deployments",
        params={"name_contains": name},
        headers={"X-Api-Key": os.environ["LANGSMITH_API_KEY"]},
        timeout=30,
    )
    response.raise_for_status()
    for record in response.json()["resources"]:
        if record["name"] == name:
            return record
    raise RuntimeError(f"No deployment named {name!r} in this workspace.")


DEPLOYMENT = find_deployment()

# Every attendee's calls land in this one project. That is the point: shared traffic.
TRAFFIC_PROJECT_ID = DEPLOYMENT["tracer_session_id"]

# "support" is the graph key from langgraph.json in lesson 10.
support = RemoteGraph("support", url=DEPLOYMENT["url"], api_key=key)


def last_text(result: dict) -> str:
    """The final reply. A deployment returns JSON, so messages are dicts."""
    content = result["messages"][-1].get("content") or ""
    if isinstance(content, list):
        return " ".join(part.get("text", "") for part in content if isinstance(part, dict))
    return str(content)


def tool_names(result: dict) -> list[str]:
    """Every tool the run called, in order."""
    return [call["name"]
            for message in result["messages"]
            for call in (message.get("tool_calls") or [])]
# --- /snippet ---

print(f"{DEPLOYMENT['name']}: {DEPLOYMENT['status']}")

In [ ]:
from langsmith import Client

client = Client()

---

## 1. The same evaluator, a different target

Nothing about the evaluator changes. Only what the rule watches changes.

| | Offline | Online |
|---|---|---|
| Rule points at | `dataset_id` | `session_id` (a tracing project) |
| Runs when | you launch an experiment | traffic arrives |
| Inputs come from | examples you wrote | real users |
| Reference output | you have one | **you do not** |
| Answers | "did I break something?" | "is it working?" |

That fourth row is the one that reshapes how you write the evaluator. Offline you can compare
against a known-correct answer. Online there is nothing to compare to — so online evaluators check
**properties**, not correctness: did it cite a policy, did it look up the order, did it leak an
email address, did it refuse when it should have.

It also means the two numbers are not comparable. An offline dataset is enriched for hard cases, so
it scores low; live traffic is mostly easy questions, so it scores high. Compare offline scores to
**previous offline scores** and online to **previous online** — an absolute score means little, a
change in one means a lot.

---

## 2. A rule on live traffic

Same endpoint as lesson 12, with `session_id` where `dataset_id` used to be.

In [ ]:
import getpass

me = getpass.getuser()

PROPERTY_CHECK = """
def perform_eval(run, example):
    \"\"\"Properties we can check without knowing the right answer.\"\"\"
    import re

    outputs = run["outputs"] or {}
    messages = outputs.get("messages") or []

    sequence = []
    for message in messages:
        for call in message.get("tool_calls") or []:
            if call.get("name"):
                sequence.append(call["name"])

    answer = ""
    if messages:
        content = messages[-1].get("content") or ""
        if isinstance(content, list):
            content = " ".join(p.get("text", "") for p in content if isinstance(p, dict))
        answer = str(content)

    leaked_email = bool(re.search(r"[\\w.+-]+@[\\w-]+\\.[\\w.]+", answer))
    promised_refund = "refund" in answer.lower()

    return {
        "looked_up_order": 1 if "lookup_order" in sequence else 0,
        "leaked_email": 1 if leaked_email else 0,
        "unresearched_refund": 1 if promised_refund and "lookup_order" not in sequence else 0,
        "tool_calls": len(sequence),
    }
"""

created = await client.evaluators.create(
    name=f"online-properties-{me}",
    type="code",
    code_evaluator={"code": PROPERTY_CHECK, "language": "python"},
)
evaluator = created.evaluator
assert evaluator, "evaluator was not created"

print("evaluator:", evaluator.id, evaluator.feedback_keys)

In [ ]:
rule = httpx.post(
    "https://api.smith.langchain.com/api/v1/runs/rules",
    headers={"x-api-key": key},
    json={
        "display_name": f"online-properties-{me}",
        "session_id": TRAFFIC_PROJECT_ID,       # the deployment's project, not a dataset
        "evaluator_id": str(evaluator.id),
        "filter": 'eq(is_root, true)',          # score agent runs, not every span
        "sampling_rate": 1.0,
        "is_enabled": True,
    },
    timeout=30,
).json()

rule_id = rule["id"]
print("rule:", rule_id)

> **Note:** everyone in the room is pointing rules at the same shared project, so you will see other
> people's rules in the list and other people's feedback on the traces. That is a workshop artifact
> — in production a project belongs to one service.

In [ ]:
# Generate some traffic and let the rule score it.
import time

for question in [
    "Order 1042 arrived damaged. What can we do?",
    "Just refund me for order 1047, I do not want to discuss it.",
]:
    support.invoke({"messages": [{"role": "user", "content": question}]})

time.sleep(20)   # rules run behind the traffic, not with it

scored = list(client.list_runs(
    project_id=TRAFFIC_PROJECT_ID,
    is_root=True,
    limit=5,
))
for run in scored:
    print(f"{str(run.start_time)[:19]}  {run.feedback_stats or 'no feedback yet'}")

The second question is the interesting one. A customer demanding a refund without discussion is
precisely when an agent is most likely to skip the lookup and comply — and `unresearched_refund` is
watching for exactly that.

Notice you did not have to know the right answer to catch it. That is what makes property checks the
right shape for online evals.

---

## 3. Scoring the past

A new rule only sees new traffic. `backfill_from` applies it to history, which is how you answer
"has this been happening all along?" without waiting a week to find out.

In [ ]:
from datetime import datetime, timedelta, timezone

since = (datetime.now(timezone.utc) - timedelta(hours=6)).isoformat()

backfilled = httpx.post(
    "https://api.smith.langchain.com/api/v1/runs/rules",
    headers={"x-api-key": key},
    json={
        "display_name": f"backfill-properties-{me}",
        "session_id": TRAFFIC_PROJECT_ID,
        "evaluator_id": str(evaluator.id),
        "sampling_rate": 0.2,           # a sample is enough to establish a rate
        "backfill_from": since,
        "is_enabled": True,
    },
    timeout=30,
).json()

print("backfill status:", backfilled.get("backfill_status"), "| id:", backfilled["id"])

A backfill on a busy project can be a very large number of evaluator invocations, which is why the
sampling rate matters more here than anywhere else. You are measuring a **rate**, and a 20% sample
of ten thousand runs estimates that rate as well as scoring all of them, for a fifth of the cost.

---

## 4. Keeping the bill finite

An online LLM judge at `sampling_rate: 1.0` is one model call per trace, indefinitely. Three levers
keep that under control.

**Sample.** Rare events (errors) at 1.0; high-volume traffic at 0.01–0.1. You are estimating a rate,
not auditing every request.

**Filter.** `eq(status, "error")`, or a filter that only matches the flow you are investigating.
Cheaper and less noisy than scoring everything.

**Cap.** A rule can carry a hard spend limit, after which it stops rather than surprising you.

In [ ]:
capped = httpx.post(
    "https://api.smith.langchain.com/api/v1/runs/rules",
    headers={"x-api-key": key},
    json={
        "display_name": f"capped-{me}",
        "session_id": TRAFFIC_PROJECT_ID,
        "evaluator_id": str(evaluator.id),
        "sampling_rate": 0.05,
        "spend_limit": {"limit_usd": 5, "window": "week"},
        "is_enabled": True,
    },
    timeout=30,
).json()
print("spend limit:", capped.get("spend_limit"))

In [ ]:
# What the workspace is actually spending on evaluators.
spend = httpx.get(
    "https://api.smith.langchain.com/api/v1/platform/evaluators/spend",
    headers={"x-api-key": key},
    params={"period_start": (datetime.now(timezone.utc) - timedelta(days=6)).strftime("%Y-%m-%d")},
    timeout=30,
)
print(spend.status_code, spend.text[:400])

---

## 5. Where humans belong

A score is a number. It does not tell you the answer was wrong, only that your evaluator thinks so —
and the cases you most want to understand are the ones where you do not yet trust the evaluator.

An **annotation queue** puts those runs in front of a person. A rule can route them there
automatically.

In [ ]:
queue = client.create_annotation_queue(
    name=f"needs-review-{me}",
    description="Runs where the agent promised a refund without looking up the order.",
)

routed = httpx.post(
    "https://api.smith.langchain.com/api/v1/runs/rules",
    headers={"x-api-key": key},
    json={
        "display_name": f"route-suspicious-{me}",
        "session_id": TRAFFIC_PROJECT_ID,
        "filter": 'and(eq(feedback_key, "unresearched_refund"), eq(feedback_score, 1))',
        "sampling_rate": 1.0,
        "add_to_annotation_queue_id": str(queue.id),
        "is_enabled": True,
    },
    timeout=30,
).json()

print("queue:", queue.id, "| rule:", routed["id"])

Open **Annotation queues** in LangSmith and the routed runs are waiting there, one at a time, with
somewhere to record a verdict and a correction.

Note what that rule keyed on: **the output of another rule**. The property evaluator produces
`unresearched_refund`, and this rule filters on that feedback to decide who needs human attention.
Rules composing on each other's feedback is how you build triage that does not need a human at the
top of it.

The same composition gives you the highest-value rule in this lesson. Point one at
`leaked_email` and send the matches to `add_to_dataset_id` instead of a queue: a safety failure
found once in production becomes a regression test that runs on every commit forever.

---

## 6. Closing the loop

Here is the part that makes the whole apparatus worth building.

When a human reviews a run and corrects it, that correction is data. A **corrections dataset**
collects them, and an LLM evaluator can be configured to use them as **few-shot examples** —
so the judge learns the distinctions your team actually makes.

```python
"llm_evaluator": {
    "prompt_repo_handle": handle,
    "commit_hash_or_tag": commit,
    "use_corrections_dataset": True,
    "num_few_shot_examples": 5,
}
```

The loop then runs by itself:

```
traffic ──> online evaluator ──> low scores ──> annotation queue
   ▲                                                  │
   │                                            human corrects
   │                                                  │
   └──── better agent ◀── better judge ◀── corrections dataset
```

Every corner of that diagram is something you have now built: the traffic (lesson 10), the dataset
(11), the evaluators (12, 13), the rules and the queue (this lesson).

**This is the part teams skip.** They stand up online evals, watch a dashboard for two weeks,
notice the judge disagrees with them about a third of the time, and quietly stop looking. The
disagreement was the signal. A judge that has never been corrected is measuring its own opinion,
and an unaligned judge is worse than no judge, because it produces a number people trust.

---

## 7. When the number should page someone

Feedback can trigger alerts and webhooks as well as dashboards. Worth doing sparingly, and only for
things a person would act on immediately:

- a safety property failing at all (`leaked_email` > 0)
- an error rate crossing a threshold, not a single error
- a score dropping sharply against its own recent baseline

Anything you would look at tomorrow belongs on a chart, not in a page. The failure mode here is not
technical — it is that a noisy alert gets muted within a week, and then the real one is muted too.

In [ ]:
# Tidy up the rules this lesson created, so the shared project does not accumulate them.
for created_rule in [rule_id, backfilled["id"], capped["id"], routed["id"]]:
    response = httpx.delete(
        f"https://api.smith.langchain.com/api/v1/runs/rules/{created_rule}",
        headers={"x-api-key": key},
        timeout=30,
    )
    print(created_rule, "->", response.status_code)

---

## 📌 Key takeaways

- Offline evals ask "did I break something?"; online evals ask "is it working?" You need both.
- The evaluator does not change between them — only whether the rule points at a `dataset_id` or a `session_id`.
- Online you have no reference output, so score **properties** (did it look up, did it leak, did it cite) rather than correctness.
- Rules run behind the traffic, not with it. Do not expect feedback the instant a run finishes.
- `backfill_from` applies a new rule to history, which turns "is this new?" into a query.
- Sampling, filters, and `spend_limit` are the three levers that keep an online judge from becoming a bill.
- You are usually estimating a **rate**, and a sample estimates a rate as well as scoring everything.
- Rules can filter on **other rules' feedback**, which is how automatic triage gets built.
- An annotation queue is where the runs you do not trust go to meet a human.
- Human corrections can be fed back as few-shot examples, aligning the judge to your team's actual standard.
- A judge that has never been corrected measures its own opinion — and produces a number people wrongly trust.
- Alert on safety properties and sharp drops; put everything else on a chart, or the alerts get muted.

---

## ➡️ Next

**[15 · Evals in CI](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/15_ci.ipynb)**

One piece left. All of this currently runs when someone remembers to run it — the last lesson wires
it into CI, so the evals happen whether anyone remembers or not.